In [1]:
import pandas as pd
import duckdb

In [2]:
con = duckdb.connect()

con.execute("""
CREATE OR REPLACE TABLE paysim AS
SELECT *
FROM read_csv_auto('../data/paysim_transactions.csv');
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
con.execute("""
SELECT COUNT(*) AS total_rows
FROM paysim;
""").df()

,total_rows
0,6362620


In [4]:
risk_score_preview = con.execute("""
WITH features AS (
    SELECT
        step,
        type,
        amount,
        oldbalanceOrg,
        newbalanceOrig,
        oldbalanceDest,
        newbalanceDest,
        isFraud,

        CASE
            WHEN type IN ('TRANSFER', 'CASH_OUT')
            THEN 1 ELSE 0
        END AS risky_type_flag,

        CASE
            WHEN amount >= 500000
            THEN 1 ELSE 0
        END AS high_amount_flag,

        CASE
            WHEN oldbalanceOrg > 0
                 AND amount >= oldbalanceOrg * 0.90
            THEN 1 ELSE 0
        END AS high_depletion_flag,

        CASE
            WHEN type IN ('TRANSFER', 'CASH_OUT')
                 AND amount > 0
                 AND ABS((newbalanceDest - oldbalanceDest) - amount) > 1
            THEN 1 ELSE 0
        END AS destination_mismatch_flag

    FROM paysim
),

scored AS (
    SELECT
        *,
        risky_type_flag * 2
        + high_amount_flag * 2
        + high_depletion_flag * 3
        + destination_mismatch_flag * 2 AS fraud_risk_score
    FROM features
)

SELECT *
FROM scored
LIMIT 50;
""").df()

risk_score_preview

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,risky_type_flag,high_amount_flag,high_depletion_flag,destination_mismatch_flag,fraud_risk_score
0,1,PAYMENT,9839.64,170136.00,160296.36,0.0,0.00,0,0,0,0,0,0
1,1,PAYMENT,1864.28,21249.00,19384.72,0.0,0.00,0,0,0,0,0,0
2,1,TRANSFER,181.00,181.00,0.00,0.0,0.00,1,1,0,1,1,7
3,1,CASH_OUT,181.00,181.00,0.00,21182.0,0.00,1,1,0,1,1,7
4,1,PAYMENT,11668.14,41554.00,29885.86,0.0,0.00,0,0,0,0,0,0
5,1,PAYMENT,7817.71,53860.00,46042.29,0.0,0.00,0,0,0,0,0,0
6,1,PAYMENT,7107.77,183195.00,176087.23,0.0,0.00,0,0,0,0,0,0
7,1,PAYMENT,7861.64,176087.23,168225.59,0.0,0.00,0,0,0,0,0,0
8,1,PAYMENT,4024.36,2671.00,0.00,0.0,0.00,0,0,0,1,0,3
9,1,DEBIT,5337.77,41720.00,36382.23,41898.0,40348.79,0,0,0,0,0,0


In [ ]:
RISK_SCORE_PREVIEW = CON.EXECUTE("""
WITH features AS (
    SELECT STEP,
        TYPE,
        AMOUNT,
        ISFRAUD,
        RISKY_TYPE_FLAG,
        HIGH_AMOUNT_FLAG,
        HIGH_DEPLETION_FLAG,
        DESTINATION_MISMATCH_FLAG,
        FRAUD_RISK_SCORE

In [13]:
risk_score_preiew = con.execute("""WITH features AS (
    SELECT
        step,
        type,
        amount,
        oldbalanceOrg,
        newbalanceOrig,
        oldbalanceDest,
        newbalanceDest,
        isFraud,
        CASE
            WHEN type IN ('TRANSFER', 'CASH_OUT')
            THEN 1 ELSE 0
        END AS risky_type_flag,
        CASE
            WHEN amount >= 500000
            THEN 1 ELSE 0
        END AS high_amount_flag,
        CASE
            WHEN oldbalanceOrg > 0
                 AND amount >= oldbalanceOrg * 0.90
            THEN 1 ELSE 0
        END AS high_depletion_flag,
        CASE
            WHEN type IN ('TRANSFER', 'CASH_OUT')
                 AND amount > 0
                 AND ABS((newbalanceDest - oldbalanceDest) - amount) > 1
            THEN 1 ELSE 0
        END AS destination_mismatch_flag
    FROM paysim
),
scored AS (
    SELECT
        *,
        risky_type_flag * 2
        + high_amount_flag * 2
        + high_depletion_flag * 3
        + destination_mismatch_flag * 2 AS fraud_risk_score
    FROM features
)
SELECT *
FROM scored limit 50;""").df()
risk_score_preiew

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,risky_type_flag,high_amount_flag,high_depletion_flag,destination_mismatch_flag,fraud_risk_score
0,1,PAYMENT,9839.64,170136.00,160296.36,0.0,0.00,0,0,0,0,0,0
1,1,PAYMENT,1864.28,21249.00,19384.72,0.0,0.00,0,0,0,0,0,0
2,1,TRANSFER,181.00,181.00,0.00,0.0,0.00,1,1,0,1,1,7
3,1,CASH_OUT,181.00,181.00,0.00,21182.0,0.00,1,1,0,1,1,7
4,1,PAYMENT,11668.14,41554.00,29885.86,0.0,0.00,0,0,0,0,0,0
5,1,PAYMENT,7817.71,53860.00,46042.29,0.0,0.00,0,0,0,0,0,0
6,1,PAYMENT,7107.77,183195.00,176087.23,0.0,0.00,0,0,0,0,0,0
7,1,PAYMENT,7861.64,176087.23,168225.59,0.0,0.00,0,0,0,0,0,0
8,1,PAYMENT,4024.36,2671.00,0.00,0.0,0.00,0,0,0,1,0,3
9,1,DEBIT,5337.77,41720.00,36382.23,41898.0,40348.79,0,0,0,0,0,0


In [16]:
fraud_risk_score = con.execute("""
WITH features AS (
    SELECT
        type,
        amount,
        oldbalanceOrg,
        newbalanceDest,
        oldbalanceDest,
        isFraud,
        CASE WHEN type IN ('TRANSFER','CASH_OUT') THEN 1 ELSE 0 END AS risky_type_flag,
        CASE WHEN amount >= 500000 THEN 1 ELSE 0 END AS high_amount_flag,
        CASE WHEN oldbalanceOrg > 0 AND amount >= oldbalanceOrg * 0.90 THEN 1 ELSE 0 END AS high_depletion_flag,
        CASE WHEN type IN ('TRANSFER','CASH_OUT') AND amount > 0 AND ABS((newbalanceDest - oldbalanceDest) - amount) > 1 THEN 1 ELSE 0 END AS destination_mismatch_flag
    FROM paysim
),
scored AS (
    SELECT
        type,
        (risky_type_flag * 2 + high_amount_flag * 2 + high_depletion_flag * 3 + destination_mismatch_flag * 2) AS fraud_risk_score,
        isFraud
    FROM features
)
SELECT
    fraud_risk_score,
    COUNT(*) AS total_transactions,
    SUM(isFraud) AS fraud_transactions,
    ROUND(SUM(isFraud) * 1.0 / COUNT(*), 6) AS fraud_rate
FROM scored
GROUP BY fraud_risk_score
ORDER BY fraud_risk_score asc;""").df()

fraud_risk_score

,fraud_risk_score,total_transactions,fraud_transactions,fraud_rate
0,0,2751765,0.0,0.000000
1,2,1283851,33.0,0.000026
2,3,814463,0.0,0.000000
3,4,274490,9.0,0.000033
4,5,967022,2075.0,0.002146
5,6,16577,135.0,0.008144
6,7,232526,4101.0,0.017637
7,9,21926,1860.0,0.084831


In [25]:
risk_tier_summary = con.execute("""
WITH features AS (
    SELECT
        type,
        amount,
        oldbalanceOrg,
        newbalanceDest,
        oldbalanceDest,
        isFraud,
        CASE WHEN type IN ('TRANSFER','CASH_OUT') THEN 1 ELSE 0 END AS risky_type_flag,
        CASE WHEN amount >= 500000 THEN 1 ELSE 0 END AS high_amount_flag,
        CASE WHEN oldbalanceOrg > 0 AND amount >= oldbalanceOrg * 0.90 THEN 1 ELSE 0 END AS high_depletion_flag,
        CASE WHEN type IN ('TRANSFER','CASH_OUT') AND amount > 0
             AND ABS((newbalanceDest - oldbalanceDest) - amount) > 1
        THEN 1 ELSE 0 END AS destination_mismatch_flag
    FROM paysim
),

scored AS (
    SELECT
        *,
        risky_type_flag * 2
        + high_amount_flag * 2
        + high_depletion_flag * 3
        + destination_mismatch_flag * 2 AS fraud_risk_score
    FROM features
),

tiered AS (
    SELECT
        *,
        CASE
            WHEN fraud_risk_score BETWEEN 0 AND 2 THEN 'Low'
            WHEN fraud_risk_score BETWEEN 3 AND 5 THEN 'Medium'
            WHEN fraud_risk_score BETWEEN 6 AND 7 THEN 'High'
            WHEN fraud_risk_score >= 8 THEN 'Critical'
        END AS risk_tier
    FROM scored
)

SELECT
    risk_tier,
    COUNT(*) AS total_transactions,
    SUM(isFraud) AS fraud_transactions,
    ROUND(SUM(isFraud) * 1.0 / COUNT(*), 6) AS fraud_rate
FROM tiered
GROUP BY risk_tier
ORDER BY fraud_rate DESC;
""").df()

risk_tier_summary.to_csv("../outputs/weighted_risk_tier_summary.csv", index=False)

risk_tier_summary

,risk_tier,total_transactions,fraud_transactions,fraud_rate
0,Critical,21926,1860.0,0.084831
1,High,249103,4236.0,0.017005
2,Medium,2055975,2084.0,0.001014
3,Low,4035616,33.0,0.000008


In [26]:
risk_tier_summary.to_csv("../outputs/weighted_risk_tier_summary.csv", index=False)

In [27]:
weighted_score_summary = con.execute("""
WITH features AS (
    SELECT
        type,
        amount,
        oldbalanceOrg,
        newbalanceDest,
        oldbalanceDest,
        isFraud,
        CASE WHEN type IN ('TRANSFER','CASH_OUT') THEN 1 ELSE 0 END AS risky_type_flag,
        CASE WHEN amount >= 500000 THEN 1 ELSE 0 END AS high_amount_flag,
        CASE WHEN oldbalanceOrg > 0 AND amount >= oldbalanceOrg * 0.90 THEN 1 ELSE 0 END AS high_depletion_flag,
        CASE WHEN type IN ('TRANSFER','CASH_OUT') AND amount > 0
             AND ABS((newbalanceDest - oldbalanceDest) - amount) > 1
        THEN 1 ELSE 0 END AS destination_mismatch_flag
    FROM paysim
),

scored AS (
    SELECT
        *,
        risky_type_flag * 2
        + high_amount_flag * 2
        + high_depletion_flag * 3
        + destination_mismatch_flag * 2 AS fraud_risk_score
    FROM features
)

SELECT
    fraud_risk_score,
    COUNT(*) AS total_transactions,
    SUM(isFraud) AS fraud_transactions,
    ROUND(SUM(isFraud) * 1.0 / COUNT(*), 6) AS fraud_rate
FROM scored
GROUP BY fraud_risk_score
ORDER BY fraud_risk_score;
""").df()

weighted_score_summary.to_csv("../outputs/weighted_risk_score_summary.csv", index=False)

weighted_score_summary

,fraud_risk_score,total_transactions,fraud_transactions,fraud_rate
0,0,2751765,0.0,0.000000
1,2,1283851,33.0,0.000026
2,3,814463,0.0,0.000000
3,4,274490,9.0,0.000033
4,5,967022,2075.0,0.002146
5,6,16577,135.0,0.008144
6,7,232526,4101.0,0.017637
7,9,21926,1860.0,0.084831
